In [ ]:
%pip install scikit-learn pandas nltk

In [1]:
import pickle
import re

import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/imakarov/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/imakarov/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/imakarov/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [2]:
# Load model and vectorizer
with open("./models/svc_model.pkl", "rb") as f:
    svc_model = pickle.load(f)
with open("./models/tfidf_vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

print("Model and vectorizer loaded successfully")

Model and vectorizer loaded successfully


In [3]:
# Load test data
df_test = pd.read_csv("./data/balanced_test.csv")
print(f"Test set: {len(df_test)} records")
df_test.head()

Test set: 22070 records


,uniqueID,drugName,condition,review,rating,date,usefulCount
0,163740,Mirtazapine,Depression,"""I've tried a few antidepressants over the yea...",10,28-Feb-12,22
1,39293,Contrave,Weight Loss,"""Contrave combines drugs that were used for al...",9,5-Mar-17,35
2,97768,Cyclafem 1 / 35,Birth Control,"""I have been on this birth control for one cyc...",9,22-Oct-15,4
3,215892,Copper,Birth Control,"""I've had the copper coil for about 3 months n...",6,6-Jun-16,1
4,71428,Levora,Birth Control,"""I was on this pill for almost two years. It d...",2,16-Apr-11,3


In [4]:
# Preprocessing function (same as in training)
STOP_WORDS = frozenset(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()
DRUG_PLACEHOLDER = "[DRUGNAME]"


def preprocess_review(text: str, drug_name: str) -> str:
    if pd.isna(text) or not str(text).strip():
        return ""

    text = str(text)

    if pd.notna(drug_name) and str(drug_name).strip():
        pattern = r"\\b" + re.escape(str(drug_name)) + r"\\b"
        text = re.sub(pattern, DRUG_PLACEHOLDER, text, flags=re.IGNORECASE)

    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()

    tokens = word_tokenize(text)
    tokens = [
        LEMMATIZER.lemmatize(tok)
        for tok in tokens
        if tok not in STOP_WORDS and len(tok) > 1
    ]

    return " ".join(tokens)


df_test["review_clean"] = df_test.apply(
    lambda row: preprocess_review(row["review"], row["drugName"]),
    axis=1,
)
print("Preprocessing done")

Preprocessing done


In [5]:
# Get predictions
X_test = tfidf.transform(df_test["review_clean"])
y_pred = svc_model.predict(X_test)
y_true = df_test["condition"].values

# Find misclassified samples
df_test["predicted"] = y_pred
df_test["correct"] = df_test["condition"] == df_test["predicted"]

errors = df_test[~df_test["correct"]].copy()
print(f"Total errors: {len(errors)} out of {len(df_test)} ({len(errors)/len(df_test)*100:.2f}%)")

Total errors: 2749 out of 22070 (12.46%)


In [7]:
# Display first 20 misclassified cases
print("\n=== First 20 Misclassified Cases ===\n")

for idx, row in errors.head(20).iterrows():
    print(f"--- Case {idx} ---")
    print(f"Drug: {row['drugName']}")
    print(f"True condition: {row['condition']}")
    print(f"Predicted: {row['predicted']}")
    print(f"Review: {row['review']}...")
    print()


=== First 20 Misclassified Cases ===

--- Case 1 ---
Drug: Contrave
True condition: Weight Loss
Predicted: Obesity
Review: "Contrave combines drugs that were used for alcohol, smoking, and opioid cessation. People lose weight on it because it also helps control over-eating. I have no doubt that most obesity is caused from sugar/carb addiction, which is just as powerful as any drug. I have been taking it for five days, and the good news is, it seems to go to work immediately. I feel hungry before I want food now. I really don't care to eat; it's just to fill my stomach. Since I have only been on it a few days, I don't know if I've lost weight (I don't have a scale), but my clothes do feel a little looser, so maybe a pound or two. I'm hoping that after a few months on this medication, I will develop healthier habits that I can continue without the aid of Contrave."...

--- Case 7 ---
Drug: Microgestin Fe 1 / 20
True condition: Acne
Predicted: Birth Control
Review: "So I was on Ginanvi f